In [ ]:
%pip install --quiet ultralytics mediapipe opencv-python "numpy<2" matplotlib


## Cell 2 - Import libraries


In [ ]:
import math
import shutil
import time
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp

from IPython.display import Video, display
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from ultralytics import YOLO

print("All libraries imported successfully!")


## Cell 3 - Define project paths

The notebook prefers the folder structure below, and automatically falls back to the
video files already present in this project.

```
3d_posebyme/
|-- models/
|   |-- yolov8n.pt
|   `-- pose_landmarker_lite.task
|-- input/
|   `-- fitness.mp4
|-- outputs/
`-- yolov8_pose_mp.ipynb
```


In [ ]:
# Change this only if your project lives in a different folder.
PROJECT_DIR = Path.cwd()

MODELS_DIR = PROJECT_DIR / "models"
YOLO_MODEL_PATH = MODELS_DIR / "yolov8n.pt"
POSE_MODEL_PATH = MODELS_DIR / "pose_landmarker_lite.task"

INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / "yolo_mediapipe_3d_output.mp4"

# Prefer input/fitness.mp4, then fall back to the videos already in this project.
INPUT_CANDIDATES = [
    INPUT_DIR / "fitness.mp4",
    INPUT_DIR / "fiteness.mp4",
    INPUT_DIR / "fiteness2.mp4",
]
INPUT_VIDEO = next((path for path in INPUT_CANDIDATES if path.exists()), INPUT_CANDIDATES[0])

print("Project directory :", PROJECT_DIR)
print("YOLO model        :", YOLO_MODEL_PATH)
print("MediaPipe model   :", POSE_MODEL_PATH)
print("Input video       :", INPUT_VIDEO)
print("Output video      :", OUTPUT_VIDEO)


## Cell 4 - Prepare model files

This cell copies `yolov8n.pt` into `models/` if it exists in the project root, and
downloads the MediaPipe Pose Landmarker Lite model when it is missing.


In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- YOLO detection model ---
if not YOLO_MODEL_PATH.exists():
    root_yolo = PROJECT_DIR / "yolov8n.pt"
    if root_yolo.exists():
        shutil.copy2(root_yolo, YOLO_MODEL_PATH)
        print(f"Copied YOLO model -> {YOLO_MODEL_PATH}")
    else:
        print("YOLO model is missing; Ultralytics will download it on first use.")

# --- MediaPipe Pose Landmarker Lite ---
POSE_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    "pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
)

if not POSE_MODEL_PATH.exists():
    print("Downloading MediaPipe pose model...")
    urllib.request.urlretrieve(POSE_MODEL_URL, POSE_MODEL_PATH)
    print(f"Saved -> {POSE_MODEL_PATH}")

print("Model setup complete.")


## Cell 5 - Check whether all required files exist


In [ ]:
if not YOLO_MODEL_PATH.exists():
    # Trigger Ultralytics' built-in download, then keep it in models/.
    YOLO("yolov8n.pt")
    downloaded = PROJECT_DIR / "yolov8n.pt"
    if downloaded.exists():
        shutil.move(str(downloaded), YOLO_MODEL_PATH)

assert YOLO_MODEL_PATH.exists(), f"YOLO model not found: {YOLO_MODEL_PATH}"
assert POSE_MODEL_PATH.exists(), f"MediaPipe model not found: {POSE_MODEL_PATH}"
assert INPUT_VIDEO.exists(), f"Input video not found: {INPUT_VIDEO}"

print("All required files found successfully!")
print("Using input video:", INPUT_VIDEO.name)


## Cell 6 - Load the YOLO detection model


In [ ]:
yolo_model = YOLO(str(YOLO_MODEL_PATH))

print("YOLO model loaded successfully!")
print("Model path:", YOLO_MODEL_PATH)


## Cell 7 - Initialize the MediaPipe Pose Landmarker

MediaPipe returns 33 body landmarks. `num_poses=1` is used because YOLO crops one
person at a time before the crop is passed to MediaPipe.


In [ ]:
base_options = python.BaseOptions(
    model_asset_path=str(POSE_MODEL_PATH)
)

options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False,
)

pose_detector = vision.PoseLandmarker.create_from_options(options)

print("MediaPipe Pose Landmarker loaded successfully!")


## Cell 8 - Define MediaPipe skeleton connections


In [ ]:
POSE_CONNECTIONS = [
    # Face
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),

    # Shoulders and torso
    (9, 10),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (11, 23), (12, 24),
    (23, 24),

    # Left hand
    (15, 17), (15, 19), (15, 21),
    (17, 19),

    # Right hand
    (16, 18), (16, 20), (16, 22),
    (18, 20),

    # Legs
    (23, 25), (25, 27),
    (27, 29), (29, 31),
    (27, 31),
    (24, 26), (26, 28),
    (28, 30), (30, 32),
    (28, 32),
]

print("Skeleton connections defined!")


## Cell 9 - Convert landmarks into image coordinates

MediaPipe returns normalized coordinates (`0.0` to `1.0`) relative to the person crop.
This function maps them back to the full frame.


In [ ]:
def map_landmarks_to_frame(landmarks, bbox, frame_h, frame_w):
    x1, y1, x2, y2 = map(int, bbox)
    crop_w = max(1, x2 - x1)
    crop_h = max(1, y2 - y1)

    mapped = []

    for landmark in landmarks:
        visibility = landmark.visibility
        if visibility is not None and visibility < 0.2:
            mapped.append(None)
            continue

        px = int(x1 + landmark.x * crop_w)
        py = int(y1 + landmark.y * crop_h)

        px = int(np.clip(px, 0, frame_w - 1))
        py = int(np.clip(py, 0, frame_h - 1))

        mapped.append((px, py))

    return mapped


## Cell 10 - Draw the 33 landmarks and skeleton


In [ ]:
def draw_pose_landmarks(frame, mapped_landmarks):
    # Skeleton connections
    for start, end in POSE_CONNECTIONS:
        if start >= len(mapped_landmarks) or end >= len(mapped_landmarks):
            continue

        p1 = mapped_landmarks[start]
        p2 = mapped_landmarks[end]

        if p1 is None or p2 is None:
            continue

        cv2.line(frame, p1, p2, (255, 255, 255), 2, cv2.LINE_AA)

    # Individual joints
    for point in mapped_landmarks:
        if point is None:
            continue
        cv2.circle(frame, point, 4, (0, 255, 255), -1, cv2.LINE_AA)

    return frame


## Cell 11 - Estimate body orientation (roll, pitch, yaw)

These angles are image-based approximations:

- **Roll** - sideways shoulder tilt.
- **Pitch** - forward/backward lean estimated from the spine direction.
- **Yaw** - left/right turn estimated from the nose offset.


In [ ]:
def estimate_body_orientation(mapped):
    def get_point(index):
        if index >= len(mapped):
            return None
        point = mapped[index]
        return None if point is None else np.array(point, dtype=float)

    left_shoulder = get_point(11)
    right_shoulder = get_point(12)
    left_hip = get_point(23)
    right_hip = get_point(24)
    nose = get_point(0)

    required = [left_shoulder, right_shoulder, left_hip, right_hip, nose]
    if any(point is None for point in required):
        return 0.0, 0.0, 0.0

    shoulder_mid = (left_shoulder + right_shoulder) / 2.0
    hip_mid = (left_hip + right_hip) / 2.0

    dx = right_shoulder[0] - left_shoulder[0]
    dy = right_shoulder[1] - left_shoulder[1]
    shoulder_width = float(np.hypot(dx, dy))

    if shoulder_width < 1.0:
        return 0.0, 0.0, 0.0

    # Roll: shoulder-line tilt. Normalise the line direction so a mirrored
    # left/right shoulder order still gives a small angle for an upright person.
    roll = math.degrees(math.atan2(dy, dx))
    if roll > 90.0:
        roll -= 180.0
    elif roll < -90.0:
        roll += 180.0

    # Pitch: spine lean away from vertical.
    spine = hip_mid - shoulder_mid
    pitch = math.degrees(math.atan2(spine[0], spine[1]))

    # Yaw: nose offset relative to the shoulder centre.
    nose_offset = nose[0] - shoulder_mid[0]
    yaw = math.degrees(math.atan2(nose_offset, shoulder_width / 2.0))

    roll = float(np.clip(roll, -45.0, 45.0))
    pitch = float(np.clip(pitch, -45.0, 45.0))
    yaw = float(np.clip(yaw, -65.0, 65.0))

    return roll, pitch, yaw


## Cell 12 - Rotation matrix for the 3D cuboid


In [ ]:
def create_rotation_matrix(roll, pitch, yaw):
    rx = math.radians(roll)
    ry = math.radians(pitch)
    rz = math.radians(yaw)

    Rx = np.array([
        [1, 0, 0],
        [0, math.cos(rx), -math.sin(rx)],
        [0, math.sin(rx), math.cos(rx)],
    ])

    Ry = np.array([
        [math.cos(ry), 0, math.sin(ry)],
        [0, 1, 0],
        [-math.sin(ry), 0, math.cos(ry)],
    ])

    Rz = np.array([
        [math.cos(rz), -math.sin(rz), 0],
        [math.sin(rz), math.cos(rz), 0],
        [0, 0, 1],
    ])

    return Rz @ Ry @ Rx


## Cell 13 - Draw the estimated 3D cuboid, axes and angle label

The box uses the shoulder width, head, hips and ankles to estimate its dimensions.
The eight corners are rotated with the roll/pitch/yaw matrix and projected back to
the image with a small fake-perspective offset.


In [ ]:
def draw_3d_person_box(frame, mapped, person_id=None, bbox=None):
    def get_point(index):
        if index >= len(mapped):
            return None
        point = mapped[index]
        return None if point is None else np.array(point, dtype=np.float32)

    left_shoulder = get_point(11)
    right_shoulder = get_point(12)
    left_hip = get_point(23)
    right_hip = get_point(24)
    left_ankle = get_point(27)
    right_ankle = get_point(28)
    nose = get_point(0)

    required = [left_shoulder, right_shoulder, left_hip, right_hip]
    if any(point is None for point in required):
        return frame

    shoulder_mid = (left_shoulder + right_shoulder) / 2.0
    hip_mid = (left_hip + right_hip) / 2.0

    shoulder_width = float(np.linalg.norm(right_shoulder - left_shoulder))
    hip_width = float(np.linalg.norm(right_hip - left_hip))

    if shoulder_width < 1.0:
        return frame

    # Use every stable body point so the cuboid encloses the whole person.
    box_points = [left_shoulder, right_shoulder, left_hip, right_hip]
    if nose is not None:
        box_points.append(nose)
    if left_ankle is not None:
        box_points.append(left_ankle)
    if right_ankle is not None:
        box_points.append(right_ankle)

    xs = np.array([point[0] for point in box_points], dtype=np.float32)
    ys = np.array([point[1] for point in box_points], dtype=np.float32)

    body_width = max(shoulder_width, hip_width, float(xs.max() - xs.min()))
    box_width = max(40.0, body_width * 1.15)
    box_depth = max(25.0, box_width * 0.55)

    top_y = float(ys.min()) - shoulder_width * 0.15
    bottom_y = float(ys.max()) + shoulder_width * 0.05
    box_height = max(60.0, bottom_y - top_y)

    center = np.array([
        float((xs.min() + xs.max()) / 2.0),
        float((top_y + bottom_y) / 2.0),
    ], dtype=np.float32)

    w = box_width / 2.0
    h = box_height / 2.0
    d = box_depth / 2.0

    vertices = np.array([
        [-w, -h, -d],
        [ w, -h, -d],
        [ w,  h, -d],
        [-w,  h, -d],
        [-w, -h,  d],
        [ w, -h,  d],
        [ w,  h,  d],
        [-w,  h,  d],
    ], dtype=np.float32)

    roll, pitch, yaw = estimate_body_orientation(mapped)
    rotation = create_rotation_matrix(roll, pitch, yaw)
    rotated = vertices @ rotation.T

    def project(point):
        px, py, pz = point
        return (
            int(round(center[0] + px + pz * 0.35)),
            int(round(center[1] + py - pz * 0.15)),
        )

    projected = [project(point) for point in rotated]

    edges = [
        (0, 1), (1, 2), (2, 3), (3, 0),
        (4, 5), (5, 6), (6, 7), (7, 4),
        (0, 4), (1, 5), (2, 6), (3, 7),
    ]

    for start, end in edges:
        cv2.line(
            frame,
            projected[start],
            projected[end],
            (255, 255, 0),
            2,
            cv2.LINE_AA,
        )

    # X / Y / Z axes
    axis_length = max(25, int(box_width * 0.30))
    origin_2d = project((0, 0, 0))

    axes = [
        ((axis_length, 0, 0), (0, 0, 255), "X"),
        ((0, -axis_length, 0), (0, 255, 0), "Y"),
        ((0, 0, axis_length), (255, 0, 0), "Z"),
    ]

    for axis, color, label in axes:
        endpoint = project(rotation @ np.array(axis, dtype=np.float32))
        cv2.arrowedLine(frame, origin_2d, endpoint, color, 2, tipLength=0.25)
        cv2.putText(
            frame,
            label,
            endpoint,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2,
            cv2.LINE_AA,
        )

    # Person ID + estimated angles
    if bbox is not None:
        text_x = int(bbox[0])
        text_y = max(25, int(bbox[1]) - 35)
    else:
        text_x = min(point[0] for point in projected)
        text_y = max(25, min(point[1] for point in projected) - 10)

    label = (
        f"ID {person_id if person_id is not None else '-'} | "
        f"R {roll:.1f}  P {pitch:.1f}  Y {yaw:.1f}"
    )

    cv2.putText(
        frame,
        label,
        (text_x, text_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 255, 255),
        2,
        cv2.LINE_AA,
    )

    return frame


## Cell 14 - Complete YOLO + ByteTrack + MediaPipe processing function

For every detected person this cell:

1. Runs YOLO + ByteTrack.
2. Crops the person (with a small margin).
3. Runs MediaPipe Pose Landmarker on the crop.
4. Maps the 33 landmarks back to the frame.
5. Draws the skeleton, the 3D cuboid, the axes and the angle label.


In [ ]:
def process_frame(frame, frame_id=0):
    frame_h, frame_w = frame.shape[:2]

    results = yolo_model.track(
        frame,
        imgsz=640,
        conf=0.4,
        classes=[0],          # Person class only
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False,
    )

    if not results:
        return frame

    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return frame

    ids = boxes.id

    for index in range(len(boxes)):
        bbox = boxes.xyxy[index].cpu().tolist()
        x1, y1, x2, y2 = map(int, bbox)

        # Keep coordinates inside the frame.
        x1 = max(0, min(x1, frame_w - 1))
        y1 = max(0, min(y1, frame_h - 1))
        x2 = max(x1 + 1, min(x2, frame_w))
        y2 = max(y1 + 1, min(y2, frame_h))

        if x2 - x1 < 10 or y2 - y1 < 20:
            continue

        person_id = int(ids[index].item()) if ids is not None else None

        # Draw the tracked person box.
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 255), 2, cv2.LINE_AA)

        # Small margin so the pose model sees the whole body.
        pad_x = int((x2 - x1) * 0.05)
        pad_y = int((y2 - y1) * 0.05)
        cx1 = max(0, x1 - pad_x)
        cy1 = max(0, y1 - pad_y)
        cx2 = min(frame_w, x2 + pad_x)
        cy2 = min(frame_h, y2 + pad_y)

        crop = frame[cy1:cy2, cx1:cx2]
        if crop.size == 0:
            continue

        rgb_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_crop)

        pose_result = pose_detector.detect(mp_image)

        if pose_result.pose_landmarks:
            mapped = map_landmarks_to_frame(
                pose_result.pose_landmarks[0],
                (cx1, cy1, cx2, cy2),
                frame_h,
                frame_w,
            )

            draw_pose_landmarks(frame, mapped)
            draw_3d_person_box(
                frame,
                mapped,
                person_id=person_id,
                bbox=bbox,
            )
        else:
            label = f"ID {person_id if person_id is not None else '-'} | no pose"
            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 165, 255),
                2,
                cv2.LINE_AA,
            )

    return frame


## Cell 15 - Read the input video information


In [ ]:
cap = cv2.VideoCapture(str(INPUT_VIDEO))

if not cap.isOpened():
    raise RuntimeError(f"Unable to open input video: {INPUT_VIDEO}")

original_fps = cap.get(cv2.CAP_PROP_FPS)
if not np.isfinite(original_fps) or original_fps <= 0:
    original_fps = 25.0

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

if frame_width <= 0 or frame_height <= 0:
    frame_width, frame_height = 1280, 720

source_width, source_height = frame_width, frame_height

# Downscale large videos for reasonable processing speed and output size.
# Set to None to keep the original resolution.
MAX_OUTPUT_SIDE = 1280
if MAX_OUTPUT_SIDE is not None:
    scale = min(1.0, MAX_OUTPUT_SIDE / max(frame_width, frame_height))
    frame_width = max(2, int(frame_width * scale) // 2 * 2)
    frame_height = max(2, int(frame_height * scale) // 2 * 2)

print("Video information")
print("-----------------")
print("Path         :", INPUT_VIDEO)
print("Source size  :", source_width, "x", source_height)
print("Output size  :", frame_width, "x", frame_height)
print("FPS          :", round(original_fps, 2))
print("Total frames :", total_frames)


## Cell 16 - Test the pipeline on one frame

Run this before processing the whole video. It is much faster for debugging.


In [ ]:
cap = cv2.VideoCapture(str(INPUT_VIDEO))
ret, test_frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("Unable to read the first frame of the input video!")

test_frame = cv2.resize(test_frame, (frame_width, frame_height))
result_frame = process_frame(test_frame.copy(), frame_id=1)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(result_frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("YOLO + MediaPipe Human Pose Estimation")
plt.show()


## Cell 17 - Process the complete video and save the output

`MAX_FRAMES` is set to `None` to process the whole video. Set it to a number such as
`300` for a quick test, then change it back to `None`.


In [ ]:
# Re-create the YOLO model so ByteTrack starts with a clean tracker state
# after the single-frame test above.
yolo_model = YOLO(str(YOLO_MODEL_PATH))

cap = cv2.VideoCapture(str(INPUT_VIDEO))
if not cap.isOpened():
    raise RuntimeError(f"Unable to open input video: {INPUT_VIDEO}")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    fourcc,
    original_fps,
    (frame_width, frame_height),
)

if not out.isOpened():
    cap.release()
    raise RuntimeError(
        "Cannot create output video. Check the OpenCV video writer/FFmpeg installation."
    )

frame_id = 0
start_time = time.time()
fps = 0.0

# Use e.g. 300 for a quick test. None processes the complete video.
MAX_FRAMES = None

print("Video processing started...")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_id += 1
        frame = cv2.resize(frame, (frame_width, frame_height))

        output_frame = process_frame(frame, frame_id)

        elapsed = time.time() - start_time
        if elapsed > 0:
            fps = frame_id / elapsed

        cv2.putText(
            output_frame,
            f"FPS: {fps:.2f}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )

        out.write(output_frame)

        if frame_id % 30 == 0:
            print(f"Processed frames: {frame_id}/{total_frames} | FPS: {fps:.2f}")

        if MAX_FRAMES is not None and frame_id >= MAX_FRAMES:
            break
finally:
    cap.release()
    out.release()

print("Video processing completed!")
print("Total frames:", frame_id)
print("Output saved at:", OUTPUT_VIDEO)


## Cell 18 - Display the output video in Jupyter Notebook


In [ ]:
if OUTPUT_VIDEO.exists():
    # Relative path works when the notebook is served from the project folder.
    video_rel_path = OUTPUT_VIDEO.relative_to(PROJECT_DIR).as_posix()
    display(Video(filename=video_rel_path, embed=False, width=900))
else:
    print("Output video not found!")


## Cell 19 - Close the MediaPipe model

Run this after all processing is finished.


In [ ]:
pose_detector.close()
print("MediaPipe resources released successfully!")


## Cell 20 - Verify the output file


In [ ]:
if OUTPUT_VIDEO.exists():
    size_mb = OUTPUT_VIDEO.stat().st_size / (1024 * 1024)
    print("Output video generated successfully!")
    print("File size:", round(size_mb, 2), "MB")
    print("Location :", OUTPUT_VIDEO)
else:
    print("Output video not found!")
